# Database Table Exporter

This notebook connects to the PostgreSQL database, lists available tables in the `public` schema, allows you to select one, and exports the complete table data to a CSV file.

In [ ]:
import pandas as pd
import psycopg2
import ipywidgets as widgets
from IPython.display import display, FileLink
import os
import sys
import logging

# Configure logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger()

# Add project root to path to allow importing db_config
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

try:
    from iris.src.initial_setup.db_config import connect_to_db
except ImportError as e:
    logger.error(f"Error importing db_config: {e}. Make sure the notebook is run from the 'notebooks' directory or adjust the path.")
    # Provide fallback connection details if import fails, prompting user
    # This part is complex to implement robustly in a notebook without user input
    # For now, we'll rely on the import working.
    raise

In [ ]:
# --- Configuration ---
DB_ENVIRONMENT = "local"  # Change to "rbc" if needed
EXPORT_DIRECTORY = "." # Export CSV to the current directory where the notebook server is running
# --- End Configuration ---

In [ ]:
conn = None
table_list = []

try:
    logger.info(f"Attempting to connect to database environment: {DB_ENVIRONMENT}")
    conn = connect_to_db(env=DB_ENVIRONMENT)

    if conn:
        logger.info("Database connection successful.")
        with conn.cursor() as cur:
            # Query to get all table names from the public schema
            cur.execute("""
                SELECT table_name
                FROM information_schema.tables
                WHERE table_schema = 'public'
                ORDER BY table_name;
            """)
            table_list = [row[0] for row in cur.fetchall()]
            logger.info(f"Found {len(table_list)} tables in public schema.")
    else:
        logger.error("Failed to establish database connection.")

except Exception as e:
    logger.error(f"An error occurred during database connection or table listing: {e}", exc_info=True)
finally:
    # Keep connection open for export, close later
    pass

# Widgets for table selection and export
if not table_list:
    print("Could not retrieve table list. Please check connection and logs.")
    table_selector = widgets.Label("No tables found or connection failed.")
    export_button = widgets.Button(description="Export Table", disabled=True)
    output_area = widgets.Output()
else:
    table_selector = widgets.Dropdown(
        options=table_list,
        description='Select Table:',
        disabled=False,
        style={'description_width': 'initial'}
    )

    export_button = widgets.Button(
        description="Export Table",
        disabled=False,
        button_style='info', # 'success', 'info', 'warning', 'danger' or ''
        tooltip='Click to export the selected table to CSV',
        icon='download'
    )

    output_area = widgets.Output()

def on_export_button_clicked(b):
    global conn # Need access to the connection
    selected_table = table_selector.value
    output_area.clear_output()

    if not selected_table:
        with output_area:
            print("Please select a table.")
        return

    if not conn or conn.closed:
        with output_area:
             print("Database connection is closed or was not established. Please rerun the connection cell.")
        # Try to reconnect
        logger.info("Attempting to reconnect...")
        conn = connect_to_db(env=DB_ENVIRONMENT)
        if not conn:
             with output_area:
                 print("Reconnection failed.")
             return
        logger.info("Reconnection successful.")

    export_filename = f"{selected_table}.csv"
    export_filepath = os.path.join(EXPORT_DIRECTORY, export_filename)

    with output_area:
        print(f"Exporting table '{selected_table}' to {export_filepath}...")
        try:
            # Construct the SQL query safely (basic quoting for table name)
            # For production use, consider more robust SQL injection prevention if table names could be untrusted
            sql_query = f'SELECT * FROM public."{selected_table}"'
            logger.info(f"Executing query: {sql_query}")

            # Use pandas to read the SQL query into a DataFrame
            df = pd.read_sql(sql_query, conn)
            logger.info(f"Fetched {len(df)} rows from table '{selected_table}'.")

            # Export DataFrame to CSV
            df.to_csv(export_filepath, index=False, encoding='utf-8')
            logger.info(f"Successfully exported table '{selected_table}' to {export_filepath}")

            print(f"Export complete! {len(df)} rows saved.")
            # Provide a download link
            display(FileLink(export_filepath))

        except Exception as e:
            logger.error(f"Error exporting table '{selected_table}': {e}", exc_info=True)
            print(f"Error exporting table: {e}")
        finally:
            # Close the connection after export attempt
            if conn and not conn.closed:
                conn.close()
                logger.info("Database connection closed.")

# Link button click event to the function
export_button.on_click(on_export_button_clicked)

# Display the widgets
display(table_selector)
display(export_button)
display(output_area)

**Instructions:**
1. Run all the cells above.
2. Select the desired table from the dropdown menu.
3. Click the "Export Table" button.
4. The table data will be saved as a CSV file in the same directory where the Jupyter Notebook server is running (`/Users/alexwday/Projects/iris-project/notebooks` if run from there).
5. A download link for the CSV file will appear in the output area below the button.